# Tree-Ring Watermark Benchmark — Kaggle
**Generation-based (Wen et al. 2023 — spectral ring in initial latents + DDIM inversion).**  
Runs the **14 WAVES attacks** from `benchmark_attacks.py`, reports PSNR / SSIM / AUROC / TPR@1% FPR.

### Before you run
1. **Settings → Accelerator → GPU T4** (or better — **not P100**, incompatible with Kaggle PyTorch)
2. **Settings → Internet → ON**
3. **Add-ons → Secrets → `HF_TOKEN`** (Hugging Face read token for SD2.1)
4. Edit **CONFIG** (`N_IMAGES`, `HF_REPO_ID` for optional upload)

Clones [wavess](https://github.com/ademladhari/wavess) (`tree-ring/` + `benchmark_attacks.py`).

**Note:** Shared watermark key across images (paper protocol). Detection score = `1 - p_value` after DDIM invert.

### Download results
Results: `/kaggle/working/wavess_results/tree_ring/`.  
Enable `UPLOAD_TO_HF` or after **Save Version**: `kaggle kernels output USER/THIS_NOTEBOOK -p ./out`

In [ ]:
# === CONFIG (edit here) ===
N_IMAGES       = 100
SEED           = 42
MODEL_ID       = 'sd2-community/stable-diffusion-2-1-base'
MODEL_DTYPE    = 'float32'     # float32 safer on T4; use float16 if you need speed + have VRAM headroom
INVERT_DTYPE   = 'float32'
NUM_STEPS      = 50
GUIDANCE       = 7.5
INVERT_STEPS   = 50

WAVES_ROOT     = '/kaggle/working/wavess'
WAVES_URL      = 'https://github.com/ademladhari/wavess.git'
TREE_RING_ROOT = f'{WAVES_ROOT}/tree-ring'
CONFIG_PATH    = f'{TREE_RING_ROOT}/configs/base.yaml'
PROMPTS_FILE   = f'{TREE_RING_ROOT}/prompts.txt'
OUTPUT_DIR     = '/kaggle/working/wavess_results/tree_ring'

# Optional: upload results to Hugging Face
UPLOAD_TO_HF   = True
HF_REPO_ID     = 'YOUR_HF_USERNAME/wavess-results'

In [ ]:
# ── 1. Clone wavess ───────────────────────────────────────────────────────────
import os, sys, subprocess
from pathlib import Path

if not os.path.isfile(f'{WAVES_ROOT}/benchmark_attacks.py'):
    print('Cloning wavess…')
    subprocess.run(['git', 'clone', '--depth', '1', WAVES_URL, WAVES_ROOT], check=True)
else:
    print('wavess already present')

SRC_ROOT = f'{TREE_RING_ROOT}/src'
for p in [WAVES_ROOT, SRC_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(TREE_RING_ROOT)

_ba = f'{WAVES_ROOT}/benchmark_attacks.py'
if not os.path.isfile(_ba):
    raise FileNotFoundError(
        f'Missing {_ba} — push benchmark_attacks.py to GitHub, then re-clone wavess.'
    )

os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('TRANSFORMERS_NO_ADVISORY_WARNINGS', '1')
print('Repo OK:', TREE_RING_ROOT)

In [ ]:
# ── 2. Install dependencies + Hugging Face login ──────────────────────────────
!pip install -q -e . "diffusers>=0.30,<0.39" "transformers>=4.36,<5" accelerate \
               pyyaml scikit-image scikit-learn tqdm scipy huggingface_hub pandas

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=False)
print('HF_TOKEN loaded')

In [ ]:
# ── 3. Imports ────────────────────────────────────────────────────────────────
import csv, json
from time import time

import numpy as np
import torch
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sklearn import metrics as sk_metrics
from tqdm.auto import tqdm

from benchmark_attacks import ATTACKS, apply_attack_rgb
from treering.config import load_config
from treering.detect import detect_watermark
from treering.fourier import circular_mask
from treering.generate import TreeRingGenerator
from treering.invert import DDIMInverter
from treering.keygen import generate_key_material

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Imports OK | GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

In [ ]:
# ── 4. Helpers ────────────────────────────────────────────────────────────────
def load_prompts(path, n):
    lines = [l.strip() for l in Path(path).read_text('utf-8').splitlines() if l.strip()]
    if len(lines) < n:
        lines = (lines * ((n // len(lines)) + 1))[:n]
    return lines[:n]

def psnr_ssim(ref, atk):
    b = atk.resize(ref.size, Image.Resampling.LANCZOS) if atk.size != ref.size else atk
    a = np.asarray(ref, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return (
        float(peak_signal_noise_ratio(a, b, data_range=255.0)),
        float(structural_similarity(a, b, channel_axis=2, data_range=255.0)),
    )

def pvalue_to_score(p_value):
    return float(np.clip(1.0 - float(p_value), 0.0, 1.0))

def auroc_tpr(pos, neg, fpr=0.01):
    y = np.concatenate([np.zeros(neg.size, np.int32), np.ones(pos.size, np.int32)])
    s = np.concatenate([neg, pos])
    auc = float(sk_metrics.roc_auc_score(y, s))
    fp, tp, _ = sk_metrics.roc_curve(y, s, pos_label=1)
    idx = np.where(fp < fpr)[0]
    return auc, float(tp[idx[-1]]) if idx.size else float(tp[0])

print('Helpers defined.')

In [ ]:
# ── 5. Load model, key, detector ──────────────────────────────────────────────
cfg = load_config(CONFIG_PATH)
cfg.model.model_id = MODEL_ID
cfg.model.dtype = MODEL_DTYPE
cfg.model.num_inference_steps = NUM_STEPS
cfg.model.guidance_scale = GUIDANCE
cfg.detection.num_inversion_steps = INVERT_STEPS

dtype_map = {
    'float16': torch.float16,
    'float32': torch.float32,
    'bfloat16': torch.bfloat16,
}
inv_dtype = dtype_map.get(INVERT_DTYPE.lower(), torch.float32)

generator = TreeRingGenerator(
    model_id=cfg.model.model_id,
    device=device,
    dtype=cfg.model.dtype,
    num_inference_steps=cfg.model.num_inference_steps,
    guidance_scale=cfg.model.guidance_scale,
)
inverter = DDIMInverter(
    model_id=cfg.model.model_id,
    device=device,
    dtype=inv_dtype,
    pipeline=generator.pipeline,
)

mask = circular_mask(
    cfg.watermark.height,
    cfg.watermark.width,
    cfg.watermark.radius,
    device=device,
)
key = generate_key_material(
    channels=cfg.watermark.channels,
    height=cfg.watermark.height,
    width=cfg.watermark.width,
    mask=mask,
    variant=cfg.watermark.key_variant,
    seed=cfg.watermark.seed,
    device=device,
)

def detect_score(pil_img):
    inverted = inverter.invert(
        images=[pil_img],
        invert_prompt=cfg.detection.invert_prompt,
        num_inversion_steps=cfg.detection.num_inversion_steps,
    )
    det = detect_watermark(
        inverted_latents=inverted,
        key=key,
        threshold=cfg.detection.threshold,
        alpha=cfg.detection.alpha,
        variance_estimation=cfg.detection.variance_estimation,
        pvalue_tail=cfg.detection.pvalue_tail,
    )[0]
    return pvalue_to_score(det.p_value)

print('Model + key loaded:', cfg.model.model_id)

In [ ]:
# ── 6. Generate watermarked + clean pairs ─────────────────────────────────────
prompts = load_prompts(PROMPTS_FILE, N_IMAGES)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

records = []
print(f'Generating {N_IMAGES} pairs (watermarked + clean)…')
t0 = time()

for i in tqdm(range(N_IMAGES), desc='generate', unit='img'):
    prompt = prompts[i]
    seed_wm = SEED + i
    seed_clean = SEED + 100_000 + i

    wm_out = generator.generate(prompts=[prompt], key=key, seed=seed_wm)
    clean_out = generator.generate(prompts=[prompt], key=None, seed=seed_clean)

    records.append({
        'prompt': prompt,
        'wm_pil': wm_out.images[0].convert('RGB'),
        'clean_pil': clean_out.images[0].convert('RGB'),
        'seed_wm': seed_wm,
        'seed_clean': seed_clean,
    })

    if (i + 1) % 10 == 0:
        elapsed = time() - t0
        rate = (i + 1) / elapsed
        eta_h = (N_IMAGES - i - 1) / rate / 3600
        print(f'  [{i+1}/{N_IMAGES}] {rate:.3f} img/s — ETA {eta_h:.1f} h', flush=True)

print(f'Done: {len(records)} pairs in {(time()-t0)/3600:.2f} h')

In [ ]:
# ── 7. Score negatives (clean images) ─────────────────────────────────────────
sanity = detect_score(records[0]['wm_pil'])
print(f'Sanity (image 0, no attack): detect_score = {sanity:.4f}')

neg_scores = []
print('Scoring negatives…')
for rec in tqdm(records, desc='negatives', unit='img'):
    neg_scores.append(detect_score(rec['clean_pil']))
neg_arr = np.asarray(neg_scores, dtype=np.float64)
print(f'Neg score mean: {neg_arr.mean():.4f}')

In [ ]:
# ── 8. Per-attack evaluation (14 WAVES attacks) ─────────────────────────────
rows = []

for spec in ATTACKS:
    psnr_vals, ssim_vals, pos_scores = [], [], []

    for i, rec in enumerate(tqdm(records, desc=spec.name, unit='img')):
        attacked = apply_attack_rgb(spec.name, rec['wm_pil'], seed=i)
        p, s = psnr_ssim(rec['wm_pil'], attacked)
        psnr_vals.append(p)
        ssim_vals.append(s)
        pos_scores.append(detect_score(attacked))

    pos_arr = np.asarray(pos_scores, dtype=np.float64)
    auc, tpr1 = auroc_tpr(pos_arr, neg_arr)

    row = {
        'method': 'tree-ring',
        'detector': 'ddim_invert_pvalue',
        'attack': spec.name,
        'description': spec.description,
        'n_images': len(records),
        'PSNR': float(np.mean(psnr_vals)),
        'SSIM': float(np.mean(ssim_vals)),
        'bit_accuracy': float('nan'),
        'AUROC': auc,
        'TPR_at_1pct_FPR': tpr1,
    }
    rows.append(row)
    print(
        f"  {spec.name:22s}: PSNR={row['PSNR']:5.1f}  SSIM={row['SSIM']:.3f}  "
        f"AUROC={auc:.3f}  TPR@1%={tpr1:.3f}"
    )

print('\nAll attacks done.')

In [ ]:
# ── 9. Save results (+ optional Hugging Face upload) ─────────────────────────
import pandas as pd
from huggingface_hub import HfApi

csv_path  = Path(OUTPUT_DIR) / 'results.csv'
json_path = Path(OUTPUT_DIR) / 'results.json'

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'method': 'tree-ring',
        'implementation': 'tree-ring/src/treering',
        'generation_based': True,
        'model_id': cfg.model.model_id,
        'n_images': len(records),
        'seed': SEED,
        'prompts_file': PROMPTS_FILE,
        'sanity_identity_score': sanity,
        'attacks': rows,
    }, f, indent=2)

print(f'Saved to {OUTPUT_DIR}')

if UPLOAD_TO_HF:
    api = HfApi(token=os.environ['HF_TOKEN'])
    for p in [csv_path, json_path]:
        remote = f'tree_ring/{p.name}'
        api.upload_file(
            path_or_fileobj=str(p),
            path_in_repo=remote,
            repo_id=HF_REPO_ID,
            repo_type='dataset',
        )
        print(f'HF upload: https://huggingface.co/datasets/{HF_REPO_ID}/resolve/main/{remote}')

df = pd.DataFrame(rows)[['attack', 'PSNR', 'SSIM', 'AUROC', 'TPR_at_1pct_FPR']]
df.columns = ['Attack', 'PSNR', 'SSIM', 'AUROC', 'TPR@1%FPR']
print(df.to_string(index=False))